# Pandas 04 — groupby, pivot, reshape

The same tiny table is used for most sections, so you can check every number by hand.

**What's in here**
1. one key, one aggregation · 2. `size` vs `count` · 3. several keys / aggregations ·
4. `transform` · 5. `filter` · 6. grouping by time parts · 7. categoricals and `observed` ·
8. `groupby().apply` · 9. `pivot_table` · 10. `pivot` vs `pivot_table` · 11. `melt` ·
12. `stack` / `unstack` · 13. `crosstab` · 14. cumulative ops in groups · 15. `first` / `last` / `nth` ·
16. rolling within groups · 17. sanity checks

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

In [2]:
df = pd.DataFrame({
    "region": ["N", "N", "S", "S", "S", "N"],
    "tariff": ["A", "B", "A", "A", "B", "A"],
    "kwh":    [10, 20, 30, 40, 50, np.nan],
})
df

,region,tariff,kwh
0,N,A,10.0
1,N,B,20.0
2,S,A,30.0
3,S,A,40.0
4,S,B,50.0
5,N,A,NaN


## 1. One key, one aggregation

`groupby("region")` splits the rows into groups. Print the groups once to see them.

In [3]:
for name, group in df.groupby("region"):
    print("group:", name)
    print(group)
    print()

group: N
  region tariff   kwh
0      N      A  10.0
1      N      B  20.0
5      N      A   NaN

group: S
  region tariff   kwh
2      S      A  30.0
3      S      A  40.0
4      S      B  50.0



In [4]:
df.groupby("region")["kwh"].sum()

region
N     30.0
S    120.0
Name: kwh, dtype: float64

N: 10 + 20 (+ NaN, skipped) = 30. S: 30 + 40 + 50 = 120.

In [5]:
df.groupby("region")["kwh"].mean()

region
N    15.0
S    40.0
Name: kwh, dtype: float64

## 2. `size` vs `count`

`size` counts rows in the group. `count` counts non-NaN values in a column.

In [6]:
print(df.groupby("region").size())
print()
print(df.groupby("region")["kwh"].count())

region
N    3
S    3
dtype: int64

region
N    2
S    3
Name: kwh, dtype: int64


N has 3 rows but only 2 kwh values. **Interview check:** after a merge, `size()` is the
number of *rows* (e.g. readings), not the number of customers.

## 3. Several keys, several aggregations

Two keys give a two-level index. `agg` takes a list, a dict, or named aggregations.

In [7]:
df.groupby(["region", "tariff"])["kwh"].sum()

region  tariff
N       A         10.0
        B         20.0
S       A         70.0
        B         50.0
Name: kwh, dtype: float64

In [8]:
df.groupby("region")["kwh"].agg(["sum", "mean", "max"])

,sum,mean,max
region,,,
N,30.0,15.0,20.0
S,120.0,40.0,50.0


In [9]:
df.groupby("region").agg(total=("kwh", "sum"), n_rows=("kwh", "size"), n_values=("kwh", "count"))

,total,n_rows,n_values
region,,,
N,30.0,3,2
S,120.0,3,3


`as_index=False` keeps the key as a column instead of the index.

In [10]:
df.groupby("region", as_index=False)["kwh"].sum()

,region,kwh
0,N,30.0
1,S,120.0


## 4. `transform` — group statistic broadcast back to every row

`agg` returns one row per group. `transform` returns one value per original row.

In [11]:
pd.DataFrame({
    "region": df["region"],
    "kwh": df["kwh"],
    "agg would give one per group ->": "",
    "transform('sum')": df.groupby("region")["kwh"].transform("sum"),
})

,region,kwh,agg would give one per group ->,transform('sum')
0,N,10.0,,30.0
1,N,20.0,,30.0
2,S,30.0,,120.0
3,S,40.0,,120.0
4,S,50.0,,120.0
5,N,NaN,,30.0


Every N row got 30, every S row got 120. This is how you compute a share of the group total.

In [12]:
df["share_of_region"] = df["kwh"] / df.groupby("region")["kwh"].transform("sum")
df

,region,tariff,kwh,share_of_region
0,N,A,10.0,0.333333
1,N,B,20.0,0.666667
2,S,A,30.0,0.250000
3,S,A,40.0,0.333333
4,S,B,50.0,0.416667
5,N,A,NaN,NaN


In [13]:
df["zscore_in_region"] = (df["kwh"] - df.groupby("region")["kwh"].transform("mean")) / df.groupby("region")["kwh"].transform("std")
df[["region", "kwh", "zscore_in_region"]]

,region,kwh,zscore_in_region
0,N,10.0,-0.707107
1,N,20.0,0.707107
2,S,30.0,-1.000000
3,S,40.0,0.000000
4,S,50.0,1.000000
5,N,NaN,NaN


## 5. `filter` — keep whole groups that satisfy a condition

Keep the rows of every region whose total kwh is above 100.

In [14]:
df.groupby("region").filter(lambda g: g["kwh"].sum() > 100)

,region,tariff,kwh,share_of_region,zscore_in_region
2,S,A,30.0,0.250000,-1.0
3,S,A,40.0,0.333333,0.0
4,S,B,50.0,0.416667,1.0


Only S rows remain (N's total is 30).

## 6. Grouping by time components — the daily load profile

Group by the hour of the timestamp. On a tiny series first.

In [15]:
ts = pd.Series([10, 20, 12, 22],
               index=pd.to_datetime(["2023-01-01 00:00", "2023-01-01 06:00",
                                     "2023-01-02 00:00", "2023-01-02 06:00"]))
ts

2023-01-01 00:00:00    10
2023-01-01 06:00:00    20
2023-01-02 00:00:00    12
2023-01-02 06:00:00    22
dtype: int64

In [16]:
ts.groupby(ts.index.hour).mean()

0    11.0
6    21.0
dtype: float64

Hour 0: (10 + 12) / 2 = 11. Hour 6: (20 + 22) / 2 = 21. Same idea on the real data:

In [17]:
real = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
profile = real.groupby(real.index.hour)["consumption_mwh"].mean().round(0)
profile

time
0     25430.0
1     24372.0
2     23875.0
3     23610.0
4     23883.0
5     24989.0
6     27230.0
7     30060.0
8     31620.0
9     32025.0
10    31781.0
11    31329.0
12    30875.0
13    30145.0
14    29786.0
15    30079.0
16    31896.0
17    34343.0
18    35363.0
19    34008.0
20    32134.0
21    30268.0
22    28044.0
23    26420.0
Name: consumption_mwh, dtype: float64

`pd.Grouper(freq="MS")` groups by calendar month start.

In [18]:
real.groupby(pd.Grouper(freq="MS"))["consumption_mwh"].mean().round(0).head(4)

time
2022-01-01 00:00:00+00:00    32486.0
2022-02-01 00:00:00+00:00    31668.0
2022-03-01 00:00:00+00:00    31312.0
2022-04-01 00:00:00+00:00    28799.0
Freq: MS, Name: consumption_mwh, dtype: float64

## 7. Categoricals and `observed`

A categorical column remembers categories that never occur. `observed=False` shows
them as empty groups; `observed=True` hides them.

In [19]:
cat = pd.DataFrame({"t": pd.Categorical(["A", "A", "B"], categories=["A", "B", "C"]), "v": [1, 2, 3]})
cat

,t,v
0,A,1
1,A,2
2,B,3


In [20]:
print(cat.groupby("t", observed=False)["v"].sum())
print()
print(cat.groupby("t", observed=True)["v"].sum())

t
A    3
B    3
C    0
Name: v, dtype: int64

t
A    3
B    3
Name: v, dtype: int64


## 8. `groupby().apply` — last resort

`apply` runs any function per group. Slow, and the result shape depends on what the
function returns. Use `agg` / `transform` when they fit.

In [21]:
def spread(g):
    return g["kwh"].max() - g["kwh"].min()

df.groupby("region").apply(spread, include_groups=False)

region
N    10.0
S    20.0
dtype: float64

## 9. `pivot_table` — two keys into a grid

Rows = region, columns = tariff, cells = the aggregation. Default `aggfunc` is **mean**.

In [22]:
df.pivot_table(index="region", columns="tariff", values="kwh", aggfunc="sum")

tariff,A,B
region,,
N,10.0,20.0
S,70.0,50.0


N/A = 10 (the NaN row is skipped), N/B = 20, S/A = 30 + 40 = 70, S/B = 50.
`margins=True` adds totals.

In [23]:
df.pivot_table(index="region", columns="tariff", values="kwh", aggfunc="sum", margins=True)

tariff,A,B,All
region,,,
N,10.0,20.0,30.0
S,70.0,50.0,120.0
All,80.0,70.0,150.0


**Pitfall:** without `aggfunc="sum"` you get the mean and may call it a total.

In [24]:
df.pivot_table(index="region", columns="tariff", values="kwh")     # means!

tariff,A,B
region,,
N,10.0,20.0
S,35.0,50.0


In [25]:
real.pivot_table(index=real.index.hour, columns=real.index.dayofweek, values="consumption_mwh").round(0).iloc[[0, 8, 18]]

time,0,1,2,3,4,5,6
time,,,,,,,
0,26013.0,26019.0,26047.0,26097.0,25913.0,23937.0,24014.0
8,32338.0,32179.0,32115.0,32297.0,32159.0,30187.0,30094.0
18,36061.0,35814.0,36026.0,36027.0,35910.0,33889.0,33839.0


## 10. `pivot` vs `pivot_table`

`pivot` does no aggregation, so it fails if a (row, column) pair appears twice.

In [26]:
try:
    df.pivot(index="region", columns="tariff", values="kwh")
except ValueError as e:
    print("ValueError:", e)

ValueError: Index contains duplicate entries, cannot reshape


In [27]:
unique_pairs = df.drop_duplicates(subset=["region", "tariff"])
unique_pairs.pivot(index="region", columns="tariff", values="kwh")

tariff,A,B
region,,
N,10.0,20.0
S,30.0,50.0


## 11. `melt` — wide back to long

Start from a 2×2 wide table.

In [28]:
wide = pd.DataFrame({"region": ["N", "S"], "A": [10, 70], "B": [20, 50]})
wide

,region,A,B
0,N,10,20
1,S,70,50


In [29]:
long = wide.melt(id_vars="region", var_name="tariff", value_name="kwh")
long

,region,tariff,kwh
0,N,A,10
1,S,A,70
2,N,B,20
3,S,B,50


Each (region, tariff) cell became one row.

## 12. `stack` / `unstack`

`stack` moves columns into the index (wide → long); `unstack` moves an index level
into columns (long → wide).

In [30]:
w = wide.set_index("region")
w

,A,B
region,,
N,10,20
S,70,50


In [31]:
stacked = w.stack()
stacked

region   
N       A    10
        B    20
S       A    70
        B    50
dtype: int64

In [32]:
stacked.unstack()

,A,B
region,,
N,10,20
S,70,50


A two-key groupby result is exactly this stacked shape, so `unstack` turns it into a grid.

In [33]:
df.groupby(["region", "tariff"])["kwh"].sum().unstack()

tariff,A,B
region,,
N,10.0,20.0
S,70.0,50.0


## 13. `crosstab` — counts of two categoricals

In [34]:
pd.crosstab(df["region"], df["tariff"])

tariff,A,B
region,,
N,2,1
S,2,1


In [35]:
pd.crosstab(df["region"], df["tariff"], normalize="index").round(2)

tariff,A,B
region,,
N,0.67,0.33
S,0.67,0.33


`normalize="index"` gives shares within each row (N: 2 of 3 are A → 0.67).

## 14. Cumulative operations within a group

`cumsum` restarts at each group; `cumcount` numbers the rows within the group from 0.

In [36]:
pd.DataFrame({
    "region": df["region"],
    "kwh": df["kwh"],
    "cumsum in region": df.groupby("region")["kwh"].cumsum(),
    "cumcount": df.groupby("region").cumcount(),
})

,region,kwh,cumsum in region,cumcount
0,N,10.0,10.0,0
1,N,20.0,30.0,1
2,S,30.0,30.0,0
3,S,40.0,70.0,1
4,S,50.0,120.0,2
5,N,NaN,NaN,2


## 15. `first` / `last` / `nth`

`first` = first non-NaN value per group; `nth(0)` = the first row whatever it holds.

In [37]:
print(df.groupby("region")["kwh"].first())
print()
print(df.groupby("region")["kwh"].last())
print()
print(df.groupby("region").nth(1))

region
N    10.0
S    30.0
Name: kwh, dtype: float64

region
N    20.0
S    50.0
Name: kwh, dtype: float64

  region tariff   kwh  share_of_region  zscore_in_region
1      N      B  20.0         0.666667          0.707107
3      S      A  40.0         0.333333          0.000000


## 16. Rolling within groups

`groupby().rolling(2)` runs the window inside each group. The result has a two-level
index (group, original row); drop the group level to align it back.

In [38]:
r = pd.DataFrame({"meter": ["m1", "m1", "m1", "m2", "m2"], "kwh": [1, 2, 3, 10, 20]})
r

,meter,kwh
0,m1,1
1,m1,2
2,m1,3
3,m2,10
4,m2,20


In [39]:
rolled = r.groupby("meter")["kwh"].rolling(2).mean()
rolled

meter   
m1     0     NaN
       1     1.5
       2     2.5
m2     3     NaN
       4    15.0
Name: kwh, dtype: float64

In [40]:
r["roll2"] = rolled.reset_index(level=0, drop=True)
r

,meter,kwh,roll2
0,m1,1,NaN
1,m1,2,1.5
2,m1,3,2.5
3,m2,10,NaN
4,m2,20,15.0


m2's first row is NaN: the window does not reach into m1. **Pitfall:** a plain
`r["kwh"].rolling(2).mean()` would average 3 and 10 across the meter boundary.

In [41]:
r["kwh"].rolling(2).mean()      # row 3 = (3 + 10) / 2, wrong

0     NaN
1     1.5
2     2.5
3     6.5
4    15.0
Name: kwh, dtype: float64

## 17. Sanity checks after groupby

Group sizes should add up to the row count, and group totals to the grand total.

In [42]:
print("rows        :", len(df))
print("sum of sizes:", df.groupby("region").size().sum())
print("grand total :", df["kwh"].sum())
print("sum of group totals:", df.groupby("region")["kwh"].sum().sum())

rows        : 6
sum of sizes: 6
grand total : 150.0
sum of group totals: 150.0


**Pitfall:** groupby drops rows whose key is NaN. Then the group totals do not add up.

In [43]:
k = pd.DataFrame({"tariff": ["A", None, "B"], "kwh": [1, 2, 3]})
print(k.groupby("tariff")["kwh"].sum())
print()
print(k.groupby("tariff", dropna=False)["kwh"].sum())

tariff
A    1
B    3
Name: kwh, dtype: int64

tariff
A      1
B      3
NaN    2
Name: kwh, dtype: int64


## Quick reference

| Want | Write |
|---|---|
| one stat per group | `df.groupby(k)[c].sum()` |
| several stats | `df.groupby(k)[c].agg(["sum", "mean"])` |
| named columns | `df.groupby(k).agg(total=(c, "sum"))` |
| rows vs values | `size()` vs `count()` |
| group stat on every row | `df.groupby(k)[c].transform("mean")` |
| keep whole groups | `df.groupby(k).filter(f)` |
| grid | `df.pivot_table(index, columns, values, aggfunc="sum")` |
| long ↔ wide | `melt` / `pivot`, `stack` / `unstack` |
| counts of two categoricals | `pd.crosstab(a, b, normalize="index")` |
| running total in group | `df.groupby(k)[c].cumsum()` |
| rolling in group | `df.groupby(k)[c].rolling(n).mean().reset_index(level=0, drop=True)` |
| keep NaN keys | `groupby(k, dropna=False)` |